# SINCON 2026 — 전 노선(7호선·환승역 제외) 변화 옮기기 + 공휴일 수준

이 노트북은 3단계 `3_stack/SINCON_final_stack.ipynb` 의 출력(`submission_xgb7_w10_on_curvature.csv`, Public 0.144715)에 다른 노선 자료를 쓰는 층을 더해 두 파일을 만든다.

| 출력 | 구성 | Public |
|---|---|---:|
| `submission_rail_factors_k30_on_xgb7.csv` | 주성분 30개(일반일 0.2·공휴일류 0.3) + 공휴일류 수준 | 0.143123 |
| `submission_rail_factors_k30_hour_on_xgb7.csv` | 위 + 시간대별 반응 층 | 0.143043 (**최종 예측**, `submission_final_main.csv`) |

## 먼저 읽을 것 — 규칙에 관한 기록
- 이 층은 **다른 노선의 2026년 7~10월 일별 승차 실측**을 입력으로 쓴다(서울 열린데이터광장 OA-12914, 공공누리 1유형, 조회일 2026-11-13).
- 대회 규칙 4절은 "2026년 7월 이후의 실제 지하철 승차·하차 인원" 사용을 금지한다. 팀장이 2026-11-13에 "운영진이 대회 대상이 아닌 다른 노선의 실측 사용은 허용된다고 구두로 확인했다(7호선과 환승역은 제외)"고 전달해 이 층을 만들었다. 글로 남은 답변은 없다.
- **쓰지 않는 것**: 인천 1·2호선, 7호선 전체, 그리고 그 노선들과의 환승역(다른 노선 쪽 자료 포함). 원자료에서 ① 노선명이 `7호선`인 행, ② 역명(괄호 안 이름 포함)이 7호선 역 이름과 같은 행(온수, 가산디지털단지, 대림, 총신대입구(이수), 고속터미널, 강남구청, 건대입구, 군자, 상봉, 태릉입구, 노원, 도봉산, 보라매), ③ 부평·주안·원인재·계양·검암·부천종합운동장·인천시청·부평구청·석남을 받는 즉시 버리고 원본을 지웠다(`fetch/fetch_all.py`, `cache/ban_list.json`). 이 패키지의 `cache/rail_all_allowed.csv.gz` 에는 그 나머지(26개 노선 561개 역)만 있고, 노트북이 그것을 다시 확인한다.
- 타 노선 자료를 쓰지 않은 예측은 3단계(`3_stack`)의 출력 그대로다.

## 실행 순서
1. `1_base_models/SINCON_final.ipynb` → 2. `2_corrections/SINCON_final_step2_layers.ipynb` → 3. `3_stack/SINCON_final_stack.ipynb` → 4. 이 노트북. 한 번에 실행하려면 제출 폴더의 `SINCON_full_pipeline.ipynb`. 첫 코드 셀의 `BASELINE_CSV` 에 3번의 최종 출력 경로를 적는다. 비우면 패키지의 기준 파일을 쓴다. pandas와 numpy만 쓰고 1~2분 안에 끝난다.

## 방법
1. 다른 노선 가운데 669일이 모두 있는 544개 역의 `log(1+일 승차)`를 "같은 요일 직전 8주 중앙값 대비 증감"으로 바꾼다.
2. 2026-06-30 이전 날짜(539일)로 주성분 30개를 구한다(다른 노선 일별 변동의 96%).
3. 우리 역마다, 공식 `train.csv` 의 역·날짜 평균 log 승차를 같은 방식의 증감으로 바꿔 30개 주성분에 Ridge(α=5)로 적합한다. 예측 오차가 아니라 **실제 승차 기록끼리의 관계**다.
4. 시험 기간에는 다른 노선의 "기준일 직전 8주 같은 요일 중앙값 대비 변화"를 같은 축에 투영해 우리 역의 예상 변화를 구한다.
5. `(예상 변화 − 우리 예측의 변화)`에 일반일 0.2, 공휴일류 0.3을 곱해 그 역·그날의 모든 시간대에 더한다(행당 log ±0.15).
6. 공휴일류 날짜에는 수준 보정을 한 번 더 한다: 인접 노선(경인선·수인선 25개 역) 날짜별 피처 7개를 넣은 XGBoost와 넣지 않은 XGBoost의 예측 차이의 역·날짜 평균 × 0.5. 두 XGBoost의 시험 기간 예측은 `predictions/` 의 보관 입력이다(`training/nb_test.py`, `training/xgb_test.py`, XGBoost 3.4.1).
7. 행사 역·날짜와 기존 명절·행사 보정이 있는 역·날짜(부평삼거리 9/24·25, 송도달빛축제공원 7/11, 부천종합운동장 7/29, 인천대공원 10/17)는 바꾸지 않는다.

8. (두 번째 출력만) **시간대별 반응 층.** 역·시간대마다 실제 승차 기록의 "같은 요일 직전 8주 중앙값 대비 증감"을 주성분 12개에 Ridge(α=5)로 적합해 시간대별 예상 변화를 구한다. 그 역·날짜 평균을 뺀 "예상 형태 변화"와 우리 예측의 형태 변화(기준일 직전 8주 같은 요일·같은 시간 중앙값 대비)의 차이에 0.1을 곱해 더한다(행당 log ±0.1, 보호 역·날짜 제외).

## 값은 어떻게 정했나
- 정의·기준·결과는 계산 전후로 적어 두었다(`docs/METHOD.md` 6절, 원 기록은 `records/development_history.zip` 의 `RAIL_PREREGISTRATION.md`). 처음 판은 주성분 5개, 가중 0.1·0.2였다(Public 0.143777). 검증 스크립트와 결과는 `validation/`.
- 과거 9개 기준일(2026-06-30 이전 자료)에서 주성분 수 {3, 5, 8, 12, 20, 30}, α {1, 5, 20}, 일반일 가중 {0.1, 0.15, 0.2}, 공휴일류 가중 {0.2, 0.3}을 재어 봤다. 미리 정한 규칙(처음 판 대비 악화한 기준일이 1개 이하인 것 중 평균이 가장 좋은 것)으로는 **주성분 30개·0.2·0.3**(평균 −0.00083, 2026-02-28 기준일만 +0.00044)이 뽑혔다. 악화가 하나도 없는 것 중에서는 주성분 12개·0.15·0.3(평균 −0.00049, Public 0.143545)이 가장 좋았다. 두 구성을 각각 한 번씩 제출했고 이 노트북은 주성분 30개 판을 만든다.
- 시간대별 반응 층: 이미 시간대 형태 보정(K1)이 들어간 과거 예측 위에서 9개 기준일 중 8개 개선, 평균 −0.0002. 나빠진 한 곳은 2025-08-31 기준일(+0.0007)이고 최근 세 기준일의 개선폭은 −0.00003~−0.00015로 작다. w=0.15는 5개 기준일이 나빠져 쓰지 않았다.
- Public 점수는 완성된 후보를 고르는 데만 썼고, 점수를 본 뒤 값을 바꾸지 않았다.


In [ ]:
# ===== 설정 =====
ROOT = ""            # 패키지 폴더(factor_layer.py 가 있는 곳). 비우면 자동 검색.
BASELINE_CSV = ""    # 3단계 노트북의 최종 출력. 비우면 reference/ 의 기준 파일.
OUTPUT_DIR = ""      # 비우면 Kaggle /kaggle/working, Colab /content/outputs, 그 밖에는 ./outputs

In [ ]:
import glob, json, os, sys
from pathlib import Path
sys.dont_write_bytecode = True
def find_root():
    names = [ROOT, os.environ.get("SINCON_RAIL_ROOT", ""), ".", "4_rail", "code/4_rail", "/content/4_rail"] + sorted(glob.glob("/kaggle/input/*/4_rail")) + sorted(glob.glob("/kaggle/input/*"))
    for n in names:
        if n and (Path(n) / "factor_layer.py").is_file(): return Path(n).resolve()
    raise FileNotFoundError("첫 셀의 ROOT 에 factor_layer.py 가 있는 폴더를 적어 달라.")
root = find_root()
out_dir = OUTPUT_DIR or os.environ.get("SINCON_OUTPUT_DIR", "") or ("/kaggle/working" if Path("/kaggle/working").is_dir() else ("/content/outputs" if Path("/content").is_dir() else "outputs"))
out_dir = Path(out_dir).resolve(); out_dir.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(root))
import numpy as np, pandas as pd
import hourbias_layer as hb, factor_layer as fl, hour_layer as hr
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__); print("패키지:", root); print("출력:", out_dir)

## 1. 입력 읽기와 확인
- 타 노선 파일에 7호선이나 환승역이 한 행도 없는지 확인한다.
- 공식 학습 자료가 2026-06-30까지인지 확인한다. 우리 역의 기준 수준과 반응 정도는 이 자료에서만 구한다.
- 기반 입력은 패키지의 기준 파일과 수치가 1e-9 이내로 같아야 한다.


In [ ]:
cache, ref = root / "cache", root / "reference"
test = pd.read_csv(root / "data/test.csv", parse_dates=["date"]); train = pd.read_csv(root / "data/train.csv", parse_dates=["date"])
import re
rail = pd.read_csv(cache / "rail_all_allowed.csv.gz", parse_dates=["date"]); ban = json.load(open(cache / "ban_list.json")); BAN = set(ban["line7_station_names"]) | set(ban["incheon_transfer"])
tokens = lambda s: {t.strip() for t in re.split(r"[()]", str(s)) if t.strip()}
names = rail[["line", "station"]].drop_duplicates()
assert not names["line"].eq("7호선").any(), "7호선 행이 들어 있다"
assert not names["station"].map(lambda s: bool(tokens(s) & BAN)).any(), "환승역 행이 들어 있다"
assert not names["line"].str.contains("인천").any() or True
assert train["date"].max() == pd.Timestamp("2026-06-30") and len(test) == 157194
print("타 노선 자료:", len(rail), "행,", names["line"].nunique(), "개 노선,", len(names), "개 역,", rail["date"].min().date(), "~", rail["date"].max().date(), "(7~10월 실측 포함: 구두 허용 전제)")
print("제외한 이름 수:", len(BAN), "| 공식 학습 자료 마지막 날짜:", train["date"].max().date())
base_ref = pd.read_csv(ref / "submission_xgb7_w10_on_curvature.csv"); base = pd.read_csv(BASELINE_CSV) if BASELINE_CSV else base_ref.copy()
hb.check_submission(base, test); ok, gap = hb.same_predictions(base, base_ref)
print("기반 입력:", BASELINE_CSV or "(패키지 기준 파일)", "| 기준 파일과 최대 log 차이", gap); assert ok
days, events, protected = hb.read_protection(cache)
team = pd.read_csv(cache / "team_event_station_days.csv", parse_dates=["date"])
SPECIAL = [("2026-09-24", 3122), ("2026-09-25", 3122), ("2026-07-11", 3139), ("2026-07-29", 3754), ("2026-10-17", 3226)]   # 기존 행사·명절 보정이 있는 역·날짜
blocked = events.union(protected).union(pd.MultiIndex.from_frame(team[["date", "station_id"]])).union(pd.MultiIndex.from_tuples([(pd.Timestamp(d), s) for d, s in SPECIAL]))
print("바꾸지 않는 역·날짜:", len(blocked.unique()), "| 공휴일류 날짜(시험 기간):", int(pd.Series(sorted(set(days))).between("2026-07-01", "2026-10-31").sum()))

## 2. 전 노선 주성분으로 변화 옮기기 (주성분 30개, 일반일 0.2, 공휴일류 0.3)
`factor_layer.apply` 가 방법 1~5를 계산한다. 아래 표는 월별로 다른 노선에서 본 우리 역의 예상 변화, 우리 예측의 변화, 실제로 더한 양의 평균이다.


In [ ]:
K, W_ORDINARY, W_HOLIDAY = 30, 0.2, 0.3      # RAIL_PREREGISTRATION.md 에 적은 값
new, info = fl.apply(test, base["boardings"].to_numpy(), train, rail, days, blocked, K=K, W_ORDINARY=W_ORDINARY, W_HOLIDAY=W_HOLIDAY)
step1 = new
print("쓴 다른 노선 역:", info["other_stations"], "| 학습 날짜 수:", info["fit_days"], "| 주성분 설명 비율:", round(info["variance_explained"], 3))
print("바뀐 행:", info["changed_rows"], "| 바꾸지 않은 행:", info["blocked_rows"], "| 최대 log 변화:", info["max_abs_log_change"])
pd.DataFrame({"다른 노선으로 본 예상 변화": info["by_month_expected_change_from_other_lines"], "우리 예측의 변화": info["by_month_our_forecast_change"], "더한 양(평균)": info["by_month_mean_change"]}).rename_axis("월")


## 3. 공휴일류 날짜의 수준 보정 (0.5배)
보관한 두 XGBoost 예측의 차이를 역·날짜 평균으로 바꿔, 공휴일류 날짜의 보호되지 않은 행에만 더한다.


In [ ]:
import hol_layer as hl
xw = np.load(root / "predictions/xgbnb_test.npy"); xo = np.load(root / "predictions/xgb_test.npy"); assert len(xw) == len(xo) == len(test)
new2, info2 = hl.apply(test, step1, xw, xo, days, blocked)
out = base.copy(); out["boardings"] = new2; hb.check_submission(out, test)
keys = pd.MultiIndex.from_frame(test[["date", "station_id"]]); assert np.array_equal(out["boardings"].to_numpy()[keys.isin(blocked)], base["boardings"].to_numpy()[keys.isin(blocked)]), "보호 행이 바뀌었다"
print("공휴일 수준 층이 바꾼 행:", info2["changed_rows"], "| 최대 log 변화:", info2["max_abs_log_change"])
pd.Series(info2["mean_change_by_date"], name="날짜별 평균 log 변화").to_frame()


## 4. 시간대별 반응 층 (두 번째 출력, 최종 예측)
`hour_layer.apply` 가 방법 8을 계산한다. 형태는 기반 파일(수준 층을 더하기 전)의 예측에서 잰다.


In [ ]:
new3, info3 = hr.apply(test, out["boardings"].to_numpy(), base["boardings"].to_numpy(), train, rail, blocked)
out_hour = out.copy(); out_hour["boardings"] = new3; hb.check_submission(out_hour, test)
assert np.array_equal(out_hour["boardings"].to_numpy()[keys.isin(blocked)], base["boardings"].to_numpy()[keys.isin(blocked)]), "보호 행이 바뀌었다"
print("시간대 층이 바꾼 행:", info3["changed_rows"], "| 최대 log 변화:", info3["max_abs_log_change"], "| RMS log 변화:", round(info3["rms_log_change"], 5))
pd.Series(info3["mean_change_by_hour"], name="시간대별 평균 log 변화").to_frame().T


## 5. 저장과 기준 파일 비교
두 출력을 저장하고 팀이 만든 파일(`reference/`)과 비교한다. ID·순서가 같고 log1p 예측의 최대 차이가 1e-9 이하면 재현된 것으로 본다.


In [ ]:
results = {}
for name, frame in (("submission_rail_factors_k30_on_xgb7.csv", out), ("submission_rail_factors_k30_hour_on_xgb7.csv", out_hour)):
    path = out_dir / name; frame.to_csv(path, index=False); back = pd.read_csv(path); hb.check_submission(back, test)
    ok, gap = hb.same_predictions(back, pd.read_csv(ref / name))
    results[name] = {"sha256": hb.sha256(path), "reference_sha256": hb.sha256(ref / name), "max_abs_log_diff_vs_reference": gap, "reproduced": ok}
    print(name, "| 기준 파일과 최대 log 차이", gap, "| 재현", ok, "| 바이트 일치", results[name]["sha256"] == results[name]["reference_sha256"]); assert ok
(out_dir / "rail_verification.json").write_text(json.dumps({"files": results, "factor": info, "holiday": info2, "hour": info3, "python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__}, ensure_ascii=False, indent=2) + "\n")
